## MCP Servers with Foundry Agents

![image](./Assets/image.png)

### Installing Required Libraries

In [ ]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

### Setting up the Environment Variables

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

### Setting up the Foundry Project Client

In [3]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Creating the OpenAI Client

In [4]:
openai_client = project_client.get_openai_client()

### Creating the MCP Server Connection ID

In [5]:
connection_id = next((connection.id for connection in project_client.connections.list() if connection.name == mcp_server_name), None)
# for connection in project_client.connections.list():
#     if connection.name == mcp_server_name:
#         connection_id = connection.id
#         break

print(f"The MCP Server Connection ID is: {connection_id}")

The MCP Server Connection ID is: /subscriptions/ac9bc6de-2e1a-4c7a-a720-b337c47d680e/resourceGroups/ai/providers/Microsoft.CognitiveServices/accounts/ai-103-studyz-resource/projects/ai-103-studyz/connections/MslearnMcpServer


### Creating the MCP Tool Spec

In [6]:
tool = MCPTool(
    server_label = "microsoft_learn_server",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
    project_connection_id=connection_id
)

### Creating the MCP Agent

In [8]:
agent = project_client.agents.create_version(
    agent_name="MCP-Agent",
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are an intelligent assistant that can interact with the Microsoft Learn MCP server to provide users with relevant learning resources and information about Microsoft technologies.",
        tools=[tool],
    )
)

print(f"Created MCP Agent with ID: {agent.id}")

Created MCP Agent with ID: MCP-Agent:1


### Creating a Conversation Object for the Agent Chat System

In [9]:
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_004410659505a39900KWR67E03JtINiKseLMwsslgZ0QpzqRy7


### Chat with the Agent

In [10]:
user_query = "Can you pls help me with the latest ai foundry SDK code samples?" 

# Can also try this query: "Find me learning paths on Azure AI services for building intelligent applications."

In [11]:
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={
        "agent": {
            "name": "MCP-Agent",
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Agent Response: {response.output_text}")

Agent Response: Yep — here are the latest **official Microsoft Learn Azure AI Foundry / Microsoft Foundry SDK code samples** I found.

## Best starting points

### 1) Get started with Microsoft Foundry SDK
Official quickstart:
https://learn.microsoft.com/azure/foundry/quickstarts/get-started-code

Microsoft Learn notes this uses **Azure AI Projects 2.x** for the **Foundry (new)** API.

### 2) SDK overview
Official SDK overview:
https://learn.microsoft.com/azure/foundry/how-to/develop/sdk-overview

Useful details from the docs:
- **Python**: `azure-ai-projects >= 2.3.0`
- **JavaScript**: `@azure/ai-projects` **2.4.0**
- Foundry SDK uses a single project endpoint like:
  `https://<resource-name>.services.ai.azure.com/api/projects/<project-name>`

---

# Latest official Python samples

## Install
```bash
pip install "azure-ai-projects>=2.3.0" azure-identity
az login
```

Microsoft’s SDK overview also says some 2.x samples require:
```bash
pip install "openai>=3.0.0"
```

---

## 1) Chat w